In [6]:
# If needed locally:
# !pip install modal

import modal


In [7]:
app = modal.App("cached-cnn-50-epochs")

data_vol = modal.Volume.from_name("plant-pictures", create_if_missing=True)
ckpt_vol = modal.Volume.from_name("plant-models", create_if_missing=True)

image = (
    modal.Image.debian_slim()
    .pip_install(
        "torch",
        "torchvision",
        "pandas",
        "scikit-learn",
        "pillow",
        "numpy",
    )
    .add_local_file(
        "dataset_and_CNN_utils.py",
        remote_path="/root/dataset_and_CNN_utils.py",
    )
)


## 2. Data split preparation

Run this once before training if `train_split.csv`, `val_split.csv`, and `classes.json` are not already present in `/data/data/`.


In [8]:
@app.function(image=image, volumes={"/data": data_vol, "/ckpt": ckpt_vol})
def prepare_data(test_size=0.15, random_state=42):
    import json
    from pathlib import Path

    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MultiLabelBinarizer

    try:
        data_vol.reload()
    except Exception:
        pass

    data_root = Path("/data/data")
    train_csv = data_root / "train.csv"
    image_dir = data_root / "train_images"

    if not train_csv.exists():
        raise FileNotFoundError(f"Missing {train_csv}")
    if not image_dir.exists():
        raise FileNotFoundError(f"Missing image directory: {image_dir}")

    df = pd.read_csv(train_csv)

    if "image" not in df.columns:
        raise ValueError(f"Expected column 'image'. Available columns: {list(df.columns)}")
    if "labels" not in df.columns:
        raise ValueError(f"Expected column 'labels'. Available columns: {list(df.columns)}")

    df["label_list"] = df["labels"].apply(lambda x: str(x).split())

    mlb = MultiLabelBinarizer()
    mlb.fit(df["label_list"])

    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(data_root / "train_split.csv", index=False)
    val_df.to_csv(data_root / "val_split.csv", index=False)

    classes = list(mlb.classes_)

    with open(data_root / "classes.json", "w") as f:
        json.dump(classes, f, indent=2)

    meta = {
        "num_train": int(len(train_df)),
        "num_val": int(len(val_df)),
        "classes": classes,
        "num_classes": int(len(classes)),
    }

    with open(data_root / "dataset_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    try:
        data_vol.commit()
    except Exception:
        pass

    return meta


## 3. Cached 50-epoch CNN training function

This function:

1. reads the split CSV files;
2. loads and resizes every image once;
3. stacks train and validation images into tensors;
4. moves those tensors to GPU;
5. trains by slicing GPU tensors directly.

The cache mode defaults to `uint8`, which is usually the safest way to fit the full dataset on a T4 GPU.


In [9]:
@app.function(
    image=image,
    volumes={"/data": data_vol, "/ckpt": ckpt_vol},
    timeout=60 * 60 * 10,
    gpu="T4",
)
def train_cached_final_cnn_50_epochs(
    num_epochs=50,
    batch_size=32,
    lr=1e-4,
    weight_decay=1e-4,
    image_size=224,
    threshold=0.5,
    cache_dtype="uint8",  # "uint8", "float16", or "float32"
):
    import json
    import math
    import time
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from PIL import Image

    from dataset_and_CNN_utils import FlexibleCNN

    try:
        data_vol.reload()
        ckpt_vol.reload()
    except Exception:
        pass

    torch.backends.cudnn.benchmark = True

    data_root = Path("/data/data")
    image_dir = data_root / "train_images"
    train_csv = data_root / "train_split.csv"
    val_csv = data_root / "val_split.csv"
    classes_path = data_root / "classes.json"

    if not train_csv.exists():
        raise FileNotFoundError(f"Missing {train_csv}. Run prepare_data() first.")
    if not val_csv.exists():
        raise FileNotFoundError(f"Missing {val_csv}. Run prepare_data() first.")
    if not classes_path.exists():
        raise FileNotFoundError(f"Missing {classes_path}. Run prepare_data() first.")
    if not image_dir.exists():
        raise FileNotFoundError(f"Missing image directory: {image_dir}")

    with open(classes_path, "r") as f:
        classes = json.load(f)

    num_classes = len(classes)
    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)

    for name, df in [("train", train_df), ("val", val_df)]:
        if "image" not in df.columns:
            raise ValueError(f"{name} CSV missing 'image'. Available columns: {list(df.columns)}")
        if "labels" not in df.columns:
            raise ValueError(f"{name} CSV missing 'labels'. Available columns: {list(df.columns)}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("This cached version is intended to run with a GPU.")

    print("Starting cached CNN training.", flush=True)
    print(f"Device: {device}", flush=True)
    print(f"Train samples: {len(train_df)}", flush=True)
    print(f"Validation samples: {len(val_df)}", flush=True)
    print(f"Number of classes: {num_classes}", flush=True)
    print(f"Image size: {image_size}", flush=True)
    print(f"Batch size: {batch_size}", flush=True)
    print(f"Cache dtype: {cache_dtype}", flush=True)

    if cache_dtype not in {"uint8", "float16", "float32"}:
        raise ValueError("cache_dtype must be one of: 'uint8', 'float16', 'float32'")

    def estimate_cache_gb(num_images, dtype_name):
        bytes_per_pixel = {"uint8": 1, "float16": 2, "float32": 4}[dtype_name]
        total_bytes = num_images * 3 * image_size * image_size * bytes_per_pixel
        return total_bytes / (1024 ** 3)

    total_images = len(train_df) + len(val_df)
    estimated_gb = estimate_cache_gb(total_images, cache_dtype)
    print(f"Estimated GPU image-cache memory: {estimated_gb:.2f} GB", flush=True)

    def encode_labels(labels_value):
        labels = str(labels_value).split()
        target = torch.zeros(num_classes, dtype=torch.float32)
        for label in labels:
            if label in class_to_idx:
                target[class_to_idx[label]] = 1.0
        return target

    def load_split_to_gpu_cache(df, split_name):
        n = len(df)

        # Keep images compact on CPU first. Resize occurs only here.
        images_cpu = torch.empty((n, 3, image_size, image_size), dtype=torch.uint8)
        labels_cpu = torch.empty((n, num_classes), dtype=torch.float32)

        start_time = time.time()
        print(f"Loading and resizing {split_name} images once...", flush=True)

        for i, row in df.iterrows():
            image_name = Path(str(row["image"])).name
            image_path = image_dir / image_name

            if not image_path.exists():
                raise FileNotFoundError(f"Missing image: {image_path}")

            with Image.open(image_path) as img:
                img = img.convert("RGB")
                img = img.resize((image_size, image_size), Image.BILINEAR)
                arr = np.asarray(img, dtype=np.uint8)

            tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
            images_cpu[i].copy_(tensor)
            labels_cpu[i].copy_(encode_labels(row["labels"]))

            if (i + 1) % 500 == 0 or (i + 1) == n:
                elapsed = time.time() - start_time
                print(
                    f"{split_name}: cached {i + 1}/{n} images "
                    f"in {elapsed:.1f}s",
                    flush=True,
                )

        print(f"Moving {split_name} cache to GPU...", flush=True)

        if cache_dtype == "uint8":
            images_gpu = images_cpu.to(device, non_blocking=True)
        elif cache_dtype == "float16":
            images_gpu = images_cpu.to(device, non_blocking=True).to(torch.float16).div_(255.0)
        else:
            images_gpu = images_cpu.to(device, non_blocking=True).to(torch.float32).div_(255.0)

        labels_gpu = labels_cpu.to(device, non_blocking=True)

        del images_cpu
        del labels_cpu
        torch.cuda.empty_cache()

        print(
            f"{split_name} GPU cache ready: images={tuple(images_gpu.shape)}, "
            f"labels={tuple(labels_gpu.shape)}, dtype={images_gpu.dtype}",
            flush=True,
        )

        return images_gpu, labels_gpu

    x_train, y_train = load_split_to_gpu_cache(train_df, "train")
    x_val, y_val = load_split_to_gpu_cache(val_df, "validation")

    # Fixed hyperparameters requested.
    num_layers = 4
    num_filters = 128
    kernel_size = 7
    dropout_rate = 0.5

    model = FlexibleCNN(
        num_layers=num_layers,
        num_filters=num_filters,
        kernel_size=kernel_size,
        num_classes=num_classes,
        batch_norm_included=False,
        image_size=image_size,
        dropout_rate=dropout_rate,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    def as_model_input(x_batch):
        # If cached as uint8, convert the selected batch to normalized float only here.
        if x_batch.dtype == torch.uint8:
            return x_batch.float().div_(255.0)
        return x_batch

    def compute_multilabel_f1_from_logits(logits, targets, threshold=0.5, eps=1e-8):
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()
        targets = targets.float()

        tp_micro = (preds * targets).sum()
        fp_micro = (preds * (1 - targets)).sum()
        fn_micro = ((1 - preds) * targets).sum()

        micro_f1 = (2 * tp_micro) / (2 * tp_micro + fp_micro + fn_micro + eps)

        tp_class = (preds * targets).sum(dim=0)
        fp_class = (preds * (1 - targets)).sum(dim=0)
        fn_class = ((1 - preds) * targets).sum(dim=0)

        macro_f1_per_class = (2 * tp_class) / (2 * tp_class + fp_class + fn_class + eps)
        macro_f1 = macro_f1_per_class.mean()

        return micro_f1.item(), macro_f1.item()

    def run_validation():
        model.eval()
        val_loss_sum = 0.0
        val_num_samples = 0
        all_logits = []
        all_targets = []

        n_val = x_val.shape[0]

        with torch.no_grad():
            for start in range(0, n_val, batch_size):
                end = min(start + batch_size, n_val)
                images = as_model_input(x_val[start:end])
                targets = y_val[start:end]

                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    logits = model(images)
                    loss = criterion(logits, targets)

                bs = end - start
                val_loss_sum += loss.item() * bs
                val_num_samples += bs

                all_logits.append(logits.detach().float().cpu())
                all_targets.append(targets.detach().float().cpu())

        val_loss = val_loss_sum / val_num_samples
        all_logits = torch.cat(all_logits, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        val_micro_f1, val_macro_f1 = compute_multilabel_f1_from_logits(
            all_logits,
            all_targets,
            threshold=threshold,
        )
        return val_loss, val_micro_f1, val_macro_f1

    ckpt_dir = Path("/ckpt")
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history_path = ckpt_dir / "cached_final_cnn_50_epoch_history.csv"
    best_model_path = ckpt_dir / "best_cached_final_cnn_50_epoch.pt"
    last_model_path = ckpt_dir / "last_cached_final_cnn_50_epoch.pt"

    history = []
    best_val_loss = float("inf")
    best_val_macro_f1 = -1.0

    n_train = x_train.shape[0]
    train_batches = math.ceil(n_train / batch_size)

    for epoch in range(num_epochs):
        epoch_start = time.time()
        print(f"\nStarting epoch {epoch + 1}/{num_epochs}", flush=True)

        model.train()
        train_loss_sum = 0.0
        train_num_samples = 0

        # Shuffle directly on GPU. No DataLoader and no file-system lookup.
        permutation = torch.randperm(n_train, device=device)

        for batch_idx, start in enumerate(range(0, n_train, batch_size)):
            end = min(start + batch_size, n_train)
            idx = permutation[start:end]

            images = as_model_input(x_train[idx])
            targets = y_train[idx]

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = end - start
            train_loss_sum += loss.item() * bs
            train_num_samples += bs

            if batch_idx % 25 == 0:
                print(
                    f"Epoch {epoch + 1}/{num_epochs} | "
                    f"Batch {batch_idx}/{train_batches} | "
                    f"train_loss={loss.item():.4f}",
                    flush=True,
                )

        train_loss = train_loss_sum / train_num_samples
        val_loss, val_micro_f1, val_macro_f1 = run_validation()

        epoch_seconds = time.time() - epoch_start

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1,
            "epoch_seconds": epoch_seconds,
            "num_layers": num_layers,
            "num_filters": num_filters,
            "kernel_size": kernel_size,
            "dropout_rate": dropout_rate,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "image_size": image_size,
            "threshold": threshold,
            "cache_dtype": cache_dtype,
        }

        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(
            f"Epoch {epoch + 1}/{num_epochs} finished | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_micro_f1={val_micro_f1:.4f} | "
            f"val_macro_f1={val_macro_f1:.4f} | "
            f"seconds={epoch_seconds:.1f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "classes": classes,
                    "num_classes": num_classes,
                    "epoch": epoch + 1,
                    "val_loss": val_loss,
                    "val_micro_f1": val_micro_f1,
                    "val_macro_f1": val_macro_f1,
                    "num_layers": num_layers,
                    "num_filters": num_filters,
                    "kernel_size": kernel_size,
                    "dropout_rate": dropout_rate,
                    "batch_size": batch_size,
                    "lr": lr,
                    "weight_decay": weight_decay,
                    "image_size": image_size,
                    "threshold": threshold,
                    "cache_dtype": cache_dtype,
                },
                best_model_path,
            )
            print(f"Saved new best model by val_loss to {best_model_path}", flush=True)

        best_val_macro_f1 = max(best_val_macro_f1, val_macro_f1)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "num_classes": num_classes,
            "epoch": num_epochs,
            "best_val_loss": best_val_loss,
            "best_val_macro_f1": best_val_macro_f1,
            "num_layers": num_layers,
            "num_filters": num_filters,
            "kernel_size": kernel_size,
            "dropout_rate": dropout_rate,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "image_size": image_size,
            "threshold": threshold,
            "cache_dtype": cache_dtype,
        },
        last_model_path,
    )

    pd.DataFrame(history).to_csv(history_path, index=False)

    try:
        ckpt_vol.commit()
    except Exception:
        pass

    return {
        "num_epochs": num_epochs,
        "best_val_loss": best_val_loss,
        "best_val_macro_f1": best_val_macro_f1,
        "history_path": str(history_path),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "num_layers": num_layers,
        "num_filters": num_filters,
        "kernel_size": kernel_size,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "lr": lr,
        "image_size": image_size,
        "cache_dtype": cache_dtype,
        "classes": classes,
    }


In [ ]:
with app.run():
    results = train_cached_final_cnn_50_epochs.remote(
        num_epochs=50,
        batch_size=32,
        lr=1e-4,
        weight_decay=1e-4,
        image_size=224,
        threshold=0.5,
        cache_dtype="uint8",
    )

    print(results)
